# Quran Search Engine

| Phase | Sections | Description |
|-------|----------|-------------|
| **Phase 1** | 1 – 5 | Setup · Load · Corpus · Preprocessing · PyTerrier Index |
| **Phase 2** | 6 – 9 | BM25/TF-IDF Ranking · AND Retrieval · RM3 PRF · BERT Expansion · Unified API |
| **Phase 3** | 10 | Professional Search Engine UI |

**Full pipeline:**
```
quran.json  (114 surahs, 6 236 ayahs)
  └─▶  Corpus Builder  ──  flatten → 6 236 ayah documents
         └─▶  Preprocessing  ──  diacritics → normalize → tokenize → stopwords → ISRI stem
                └─▶  PyTerrier Index  ──  IterDictIndexer (on-disk Terrier index)
                       └─▶  BM25 / TF-IDF ranked retrieval  (OR union)
                              └─▶  AND retrieval  (Terrier AND controls)
                                     └─▶  RM3 PRF  (relevance model expansion)
                                            └─▶  BERT Semantic Expansion  (multilingual embeddings)
                                                   └─▶  Professional Search Engine UI
```

---
## Section 1 — Setup & Dependencies

In [45]:
!pip install nltk pyarabic sentence-transformers python-terrier --quiet

In [46]:
import json
import re
import os
import math
import urllib.request
import numpy as np
import pandas as pd
from collections import defaultdict
from typing import List, Dict, Tuple, Set

import nltk
from nltk.stem.isri import ISRIStemmer
nltk.download('stopwords', quiet=True)

import pyterrier as pt
if not pt.started():
    pt.init()

print('All imports successful')

All imports successful


/tmp/ipykernel_15505/2296735129.py:16: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


In [47]:
import urllib.request
if not os.path.exists('/content/quran.json') or os.path.getsize('/content/quran.json') < 100000:
    urllib.request.urlretrieve('https://cdn.jsdelivr.net/npm/quran-json@3.0.0/dist/quran.json', '/content/quran.json')

---
## Section 2 — Load Data

Loads `quran.json` from a local path **or** downloads it from GitHub if not present.

Expected JSON structure:
```json
[
  {
    "id": 1, "name": "الفاتحة", "transliteration": "Al-Fatiha",
    "type": "meccan", "total_verses": 7,
    "verses": [ {"id": 1, "text": "بِسۡمِ ٱللَّهِ..."}, ... ]
  }, ...
]
```

In [48]:
PROJECT_DIR = '/content'
QURAN_PATH = os.path.join(PROJECT_DIR, 'quran.json')
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'quran_ir_output')

def load_quran(path: str) -> List[Dict]:
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    assert isinstance(data, list) and len(data) == 114
    return data

def validate_structure(data: List[Dict]) -> None:
    for surah in data:
        for verse in surah.get('verses', []):
            assert verse.get('text', '').strip(), f"Empty verse: [{surah['id']}:{verse['id']}]"

raw_data = load_quran(QURAN_PATH)
validate_structure(raw_data)
print(f"Loaded: {len(raw_data)} surahs, {sum(s['total_verses'] for s in raw_data):,} ayahs")

Loaded: 114 surahs, 6,236 ayahs


In [49]:
# Print first surah as a structural sample
sample = raw_data[0]
print(f"Surah  : {sample['id']} — {sample['name']} ({sample.get('transliteration','')}")
print(f"Type   : {sample.get('type','')}   |   Verses: {sample['total_verses']}")
print()
for v in sample['verses']:
    print(f"  [{sample['id']}:{v['id']}]  {v['text']}")

Surah  : 1 — الفاتحة (Al-Fatihah
Type   : meccan   |   Verses: 7

  [1:1]  بِسۡمِ ٱللَّهِ ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ
  [1:2]  ٱلۡحَمۡدُ لِلَّهِ رَبِّ ٱلۡعَٰلَمِينَ
  [1:3]  ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ
  [1:4]  مَٰلِكِ يَوۡمِ ٱلدِّينِ
  [1:5]  إِيَّاكَ نَعۡبُدُ وَإِيَّاكَ نَسۡتَعِينُ
  [1:6]  ٱهۡدِنَا ٱلصِّرَٰطَ ٱلۡمُسۡتَقِيمَ
  [1:7]  صِرَٰطَ ٱلَّذِينَ أَنۡعَمۡتَ عَلَيۡهِمۡ غَيۡرِ ٱلۡمَغۡضُوبِ عَلَيۡهِمۡ وَلَا ٱلضَّآلِّينَ


---
## Section 3 — Build Corpus

Each ayah becomes one document:
```json
{ "doc_id": 1, "surah_id": 1, "surah_name": "الفاتحة", "ayah_id": 1, "text": "بِسۡمِ ٱللَّهِ..." }
```

In [50]:
def build_corpus(data: List[Dict]) -> List[Dict]:
    """Flatten all surahs into a list of ayah documents with sequential doc_id."""
    corpus: List[Dict] = []
    doc_id = 1
    for surah in data:
        for verse in surah['verses']:
            text = verse['text'].strip()
            if not text:
                continue
            corpus.append({
                'doc_id'    : doc_id,
                'surah_id'  : surah['id'],
                'surah_name': surah['name'],
                'transliteration': surah.get('transliteration', ''),
                'surah_type': surah.get('type', ''),
                'ayah_id'   : verse['id'],
                'text'      : text,
            })
            doc_id += 1
    return corpus


corpus = build_corpus(raw_data)

print(f'Corpus built: {len(corpus):,} documents')
print()
print('Sample documents:')
for doc in corpus[:4]:
    print(f"  doc_id={doc['doc_id']:4d}  [{doc['surah_id']}:{doc['ayah_id']}]  {doc['text']}")

Corpus built: 6,236 documents

Sample documents:
  doc_id=   1  [1:1]  بِسۡمِ ٱللَّهِ ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ
  doc_id=   2  [1:2]  ٱلۡحَمۡدُ لِلَّهِ رَبِّ ٱلۡعَٰلَمِينَ
  doc_id=   3  [1:3]  ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ
  doc_id=   4  [1:4]  مَٰلِكِ يَوۡمِ ٱلدِّينِ


---
## Section 4 — Arabic Preprocessing

| Step | Function | What it does |
|------|----------|--------------|
| 1 | `remove_diacritics` | Strip tashkeel, harakat, tatweel (U+064B–U+065F, U+0640, U+0670, U+06D6–U+06ED) |
| 2 | `normalize_arabic` | أ/إ/آ/ٱ→ا · ة→ه · ى/ئ→ي · ؤ→و |
| 3 | `tokenize` | Whitespace split, remove non-Arabic characters |
| 4 | `remove_stopwords` | Drop Arabic function words (691 terms incl. NLTK list) |
| 5 | `stem_words` | ISRI light Arabic stemmer (prefix/suffix stripping) |
| — | `preprocess` | Full pipeline in one call |

In [51]:
# ── Compiled regex patterns ───────────────────────────────────────────────────
_DIACRITICS_RE = re.compile(
    r'[\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E4\u06E7\u06E8\u06EA-\u06ED\u0640]'
)
_NON_ARABIC_RE = re.compile(r'[^\u0621-\u064A\s]')


def remove_diacritics(text: str) -> str:
    """Remove all tashkeel, harakat, and tatweel."""
    return _DIACRITICS_RE.sub('', text)


def normalize_arabic(text: str) -> str:
    """Normalise Arabic letter variants to canonical base forms."""
    text = re.sub(r'[أإآٱ]', 'ا', text)
    text = re.sub(r'ة',       'ه', text)
    text = re.sub(r'[ىئ]',   'ي', text)
    return re.sub(r'ؤ',       'و', text)


def tokenize(text: str) -> List[str]:
    """Replace non-Arabic characters with spaces then split on whitespace."""
    return [tok for tok in _NON_ARABIC_RE.sub(' ', text).split() if tok]


# ── Arabic Stopwords ──────────────────────────────────────────────────────────
_HARDCODED_SW: Set[str] = {
    'من','إلى','عن','على','في','مع','عند','بين','قبل','بعد','فوق','تحت',
    'حول','خلال','إلا','غير','دون','ضد','نحو','حتى','منذ','رغم',
    'هو','هي','هم','هن','نحن','أنت','أنتم','أنتن','أنا','هما','أنتما',
    'هذا','هذه','ذلك','تلك','هؤلاء','أولئك','هنا','هناك','ثمة',
    'الذي','التي','الذين','اللاتي','اللتان','اللذان','اللواتي',
    'و','أو','أم','لكن','بل','ثم','فـ','واو','إذ','إذا','لو','لولا','لما',
    'أن','إن','كأن','لأن','لكي','كي','حين','عندما','بينما','منذ',
    'قد','لم','لن','لا','ما','ليس','ليست','ألا','أما','إما',
    'ها','يا','أيها','أيتها','سوف','سـ',
    'كان','كانت','كانوا','يكون','تكون','يكونوا','تكونوا',
    'صار','صارت','أصبح','أصبحت','بات','باتت','ظل','ظلت','بقي','بقيت',
    'له','لها','لهم','لنا','لكم','لكن',
    'به','بها','بهم','بنا','بكم',
    'منه','منها','منهم','منا','منكم',
    'عليه','عليها','عليهم','علينا','عليكم',
    'إليه','إليها','إليهم','إلينا','إليكم',
    'عنه','عنها','عنهم','عنا','عنكم',
    'فيه','فيها','فيهم','فينا','فيكم',
    'معه','معها','معهم','معنا','معكم',
    'ذو','ذات','ذوو','ذوات','أي','أيها','كم','أين','كيف','متى','لماذا',
    'كل','كلا','كلما','جميع','بعض','كثير','قليل',
    'أيضا','جدا','نفس','أحد','إحدى',
    'وما','وهو','وهي','وهم','وكان','وكانت','وقد','ولا','ولم','وإن','وأن',
    'فما','فهو','فهي','فهم','فقد','فلا','فلم','فكان','فكانت','فإن','فأن',
    'إنه','إنها','إنهم','إنك','إننا','إنكم',
    'انه','انها','انهم','انك','اننا','انكم','انما',
}

def _nw(w: str) -> str:
    return normalize_arabic(remove_diacritics(w))

ARABIC_STOPWORDS: Set[str] = {_nw(w) for w in _HARDCODED_SW}
try:
    from nltk.corpus import stopwords as _nltk_sw
    ARABIC_STOPWORDS |= {_nw(w) for w in _nltk_sw.words('arabic')}
    print(f'NLTK stopwords merged  →  total: {len(ARABIC_STOPWORDS)} stopwords')
except Exception:
    print(f'Using hardcoded list only: {len(ARABIC_STOPWORDS)} stopwords')


def remove_stopwords(tokens: List[str]) -> List[str]:
    """Filter Arabic stopwords. Input must already be normalised."""
    return [t for t in tokens if t not in ARABIC_STOPWORDS]


_stemmer = ISRIStemmer()

def stem_words(tokens: List[str]) -> List[str]:
    """Apply ISRI light Arabic stemmer."""
    return [_stemmer.stem(t) for t in tokens]


def preprocess(text: str, apply_stemming: bool = True) -> Tuple[str, List[str]]:
    """
    Full Arabic preprocessing pipeline.
    Returns: (normalized_text, tokens)
    """
    text = remove_diacritics(text)
    text = normalize_arabic(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    if apply_stemming:
        tokens = stem_words(tokens)
    return text, tokens


print('Preprocessing functions defined')

NLTK stopwords merged  →  total: 691 stopwords
Preprocessing functions defined


In [52]:
# ── Step-by-step demo on Al-Fatiha verse 1 ────────────────────────────────────
demo_text = corpus[0]['text']
print('Original         :', demo_text)
step1 = remove_diacritics(demo_text)
print('After diacritics :', step1)
step2 = normalize_arabic(step1)
print('After normalize  :', step2)
step3 = tokenize(step2)
print('After tokenize   :', step3)
step4 = remove_stopwords(step3)
print('After stopwords  :', step4)
step5 = stem_words(step4)
print('After stemming   :', step5)

Original         : بِسۡمِ ٱللَّهِ ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ
After diacritics : بسم ٱلله ٱلرحمن ٱلرحيم
After normalize  : بسم الله الرحمن الرحيم
After tokenize   : ['بسم', 'الله', 'الرحمن', 'الرحيم']
After stopwords  : ['بسم', 'الله', 'الرحمن', 'الرحيم']
After stemming   : ['بسم', 'الل', 'رحم', 'رحم']


In [53]:
def process_corpus(corpus: List[Dict], apply_stemming: bool = True) -> List[Dict]:
    """Enrich every document with normalized_text and tokens fields."""
    processed: List[Dict] = []
    for doc in corpus:
        normed, tokens = preprocess(doc['text'], apply_stemming=apply_stemming)
        processed.append({**doc, 'normalized_text': normed, 'tokens': tokens})
    return processed


processed_corpus = process_corpus(corpus, apply_stemming=True)

print(f'Preprocessing complete: {len(processed_corpus):,} documents')
print()
for doc in processed_corpus[:3]:
    print(f"  [{doc['surah_id']}:{doc['ayah_id']}]")
    print(f"    Original        : {doc['text']}")
    print(f"    Normalized text : {doc['normalized_text']}")
    print(f"    Tokens          : {doc['tokens']}")
    print()

Preprocessing complete: 6,236 documents

  [1:1]
    Original        : بِسۡمِ ٱللَّهِ ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ
    Normalized text : بسم الله الرحمن الرحيم
    Tokens          : ['بسم', 'الل', 'رحم', 'رحم']

  [1:2]
    Original        : ٱلۡحَمۡدُ لِلَّهِ رَبِّ ٱلۡعَٰلَمِينَ
    Normalized text : الحمد لله رب العلمين
    Tokens          : ['حمد', 'لله', 'علم']

  [1:3]
    Original        : ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ
    Normalized text : الرحمن الرحيم
    Tokens          : ['رحم', 'رحم']



In [54]:
# ── Corpus statistics ─────────────────────────────────────────────────────────
all_tokens  = [tok for doc in processed_corpus for tok in doc['tokens']]
token_freq  = defaultdict(int)
for t in all_tokens: token_freq[t] += 1

empty_docs = [d for d in processed_corpus if not d['tokens']]
top_10     = sorted(token_freq.items(), key=lambda x: x[1], reverse=True)[:10]

print('Corpus Statistics')
print('─' * 45)
print(f'  Total documents         : {len(processed_corpus):,}')
print(f'  Total tokens (all docs) : {len(all_tokens):,}')
print(f'  Vocabulary size         : {len(token_freq):,} unique terms')
print(f'  Avg tokens per document : {len(all_tokens)/len(processed_corpus):.1f}')
print(f'  Documents with 0 tokens : {len(empty_docs)}')
print()
print(f'  Top 10 terms after preprocessing')
print(f'  {"Term":<20} Freq')
print('  ' + '─' * 28)
for term, freq in top_10:
    print(f'  {term:<20} {freq:,}')

Corpus Statistics
─────────────────────────────────────────────
  Total documents         : 6,236
  Total tokens (all docs) : 50,654
  Vocabulary size         : 3,942 unique terms
  Avg tokens per document : 8.1
  Documents with 0 tokens : 9

  Top 10 terms after preprocessing
  Term                 Freq
  ────────────────────────────
  الل                  2,227
  علم                  682
  قال                  664
  كفر                  524
  رسل                  512
  ارض                  459
  يوم                  396
  عذب                  371
  عمل                  360
  بين                  340


---
## Section 5 — Build PyTerrier Index

Replaces the manual inverted index with a PyTerrier `IterDictIndexer` on-disk index.

| Saved artifact | Contents | Used in |
|----------------|----------|---------|
| `pt_index/` | PyTerrier on-disk Terrier index | Phase 2 retrieval |
| `c_quran.json` | Full preprocessed corpus | Phase 2 / 3 |
| `vocab.json` | term → integer ID | Phase 2 vectorisation |
| `doc_metadata.json` | Lightweight display table | Phase 3 UI |

In [55]:
INDEX_DIR = os.path.join(OUTPUT_DIR, 'pt_index')
os.makedirs(OUTPUT_DIR, exist_ok=True)


def build_pt_index(processed_corpus: List[Dict], index_dir: str) -> pt.IndexRef:
    """
    Build a PyTerrier on-disk index from the preprocessed corpus.
    Tokens are pre-processed (stemmed, stopwords removed), so PyTerrier
    uses WhitespaceTokeniser with stemming/stopwords disabled.
    """
    def _doc_iter():
        for doc in processed_corpus:
            yield {
                'docno': str(doc['doc_id']),
                'text' : ' '.join(doc['tokens']),
            }

    indexer = pt.IterDictIndexer(
        index_dir,
        overwrite   = True,
        meta        = {'docno': 20},
        tokeniser   = 'WhitespaceTokeniser',
        stemmer     = 'none',
        stopwords   = 'none',
    )
    return indexer.index(_doc_iter())


index_ref = build_pt_index(processed_corpus, INDEX_DIR)
index     = pt.IndexFactory.of(index_ref)
lex       = index.getLexicon()
stats     = index.getCollectionStatistics()
num_terms = stats.getNumberOfUniqueTerms()
num_docs  = stats.getNumberOfDocuments()

print(f'PyTerrier index built at: {INDEX_DIR}')
print(f'  Documents    : {num_docs:,}')
print(f'  Unique terms : {num_terms:,}')
print(f'  Total tokens : {stats.getNumberOfTokens():,}')

23:41:18.970 [ForkJoinPool-4-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (4134) - further warnings are suppressed
23:41:19.542 [ForkJoinPool-4-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 9 empty documents
PyTerrier index built at: /content/quran_ir_output/pt_index
  Documents    : 6,236
  Unique terms : 3,942
  Total tokens : 50,654


In [56]:
def lookup_term(raw_term: str, top_k: int = 5) -> None:
    """Preprocess a raw term and inspect its posting list via the PyTerrier index."""
    _, stemmed = preprocess(raw_term)
    if not stemmed:
        print(f"'{raw_term}' → removed as stopword/empty after preprocessing")
        return
    term = stemmed[0]
    try:
        lee     = lex[term]
        inv     = index.getInvertedIndex()
        posting = inv.getPostings(lee)
        entries = {}
        count   = 0
        for p in posting:
            if count >= top_k:
                break
            entries[str(p.getId())] = p.getFrequency()
            count += 1
        print(f"'{raw_term}'  →  stemmed: '{term}'  |  df={lee.getDocumentFrequency()}  "
              f"|  postings (first {top_k}): {entries}")
    except Exception:
        print(f"'{raw_term}'  →  stemmed: '{term}'  →  NOT FOUND in index")


for word in ['الله', 'رحمة', 'نور', 'كتاب', 'صلاة', 'يوم', 'قلب']:
    lookup_term(word)

'الله'  →  stemmed: 'الل'  |  df=1618  |  postings (first 5): {'0': 1, '13': 1, '15': 1, '16': 1, '21': 1}
'رحمة'  →  stemmed: 'رحم'  |  df=293  |  postings (first 5): {'0': 2, '2': 2, '43': 1, '60': 1, '70': 1}
'نور'  →  stemmed: 'نور'  |  df=28  |  postings (first 5): {'263': 2, '666': 1, '683': 1, '684': 1, '789': 1}
'كتاب'  →  stemmed: 'كتب'  |  df=279  |  postings (first 5): {'8': 1, '50': 1, '59': 1, '84': 1, '85': 3}
'صلاة'  →  stemmed: 'صله'  |  df=64  |  postings (first 5): {'9': 1, '49': 1, '51': 1, '89': 1, '116': 1}
'يوم'  →  stemmed: 'يوم'  |  df=367  |  postings (first 5): {'3': 1, '9': 1, '10': 1, '12': 1, '54': 1}
'قلب'  →  stemmed: 'قلب'  |  df=153  |  postings (first 5): {'13': 1, '16': 1, '80': 1, '94': 1, '99': 1}


In [57]:
def _save_json(obj: object, fname: str) -> None:
    path = os.path.join(OUTPUT_DIR, fname)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    print(f'  Saved {fname:<35} {os.path.getsize(path)/1024:8.1f} KB')


print('Saving Phase 1 artifacts...')
_save_json(processed_corpus, 'c_quran.json')
_save_json(
    {entry.getKey(): idx for idx, entry in enumerate(lex)},
    'vocab.json'
)
_save_json(
    [{'doc_id': d['doc_id'], 'surah_id': d['surah_id'],
      'surah_name': d['surah_name'], 'transliteration': d.get('transliteration',''),
      'surah_type': d.get('surah_type',''), 'ayah_id': d['ayah_id'], 'text': d['text']}
     for d in processed_corpus],
    'doc_metadata.json'
)
print(f'\nPyTerrier index + JSON artifacts saved to ./{OUTPUT_DIR}/')

Saving Phase 1 artifacts...
  Saved c_quran.json                          4193.0 KB
  Saved vocab.json                              69.6 KB
  Saved doc_metadata.json                     2390.7 KB

PyTerrier index + JSON artifacts saved to .//content/quran_ir_output/


---
## Section 6 — BM25 / TF-IDF Ranked Retrieval via PyTerrier

Replaces the manual TF-IDF scoring loop with `pt.BatchRetrieve`.

| Pipeline | Model | Mode |
|----------|-------|------|
| `_tfidf` | `TF_IDF` | OR (union) — default |
| `_bm25` | `BM25` | OR (union) |
| `_and_br` | `BM25` + AND control | AND (intersection) |

In [58]:
# ── Lookup structures ─────────────────────────────────────────────────────────
DOC_LOOKUP = {d['doc_id']: d for d in processed_corpus}

# ── PyTerrier retrieval pipelines ─────────────────────────────────────────────
_bm25   = pt.BatchRetrieve(index_ref, wmodel='BM25',   num_results=50)
_tfidf  = pt.BatchRetrieve(index_ref, wmodel='TF_IDF', num_results=50)
_and_br = pt.BatchRetrieve(index_ref, wmodel='BM25',   num_results=50,
                            controls={'and': True})


def _preprocess_query(query: str) -> str:
    """Return whitespace-joined preprocessed tokens for PyTerrier."""
    _, tokens = preprocess(query)
    return ' '.join(tokens)


def _pt_results_to_list(res_df: pd.DataFrame, top_k: int) -> List[Dict]:
    """Convert a PyTerrier results DataFrame to the project result-dict format."""
    rows = []
    for _, row in res_df.head(top_k).iterrows():
        doc_id = int(row['docno'])
        doc    = DOC_LOOKUP.get(doc_id, {})
        rows.append({**doc, 'score': round(float(row['score']), 4)})
    return rows


def search_or(query: str, top_k: int = 10) -> List[Dict]:
    """TF-IDF OR retrieval via PyTerrier."""
    q_str = _preprocess_query(query)
    if not q_str.strip():
        print('Query empty after preprocessing.')
        return []
    return _pt_results_to_list(_tfidf.search(q_str), top_k)


def search_and(query: str, top_k: int = 10) -> List[Dict]:
    """
    BM25 AND retrieval via PyTerrier.
    Falls back to OR when the intersection is empty.
    """
    q_str = _preprocess_query(query)
    if not q_str.strip():
        print('Query empty after preprocessing.')
        return []
    res = _and_br.search(q_str)
    if res.empty:
        print('No AND match — falling back to OR.')
        return search_or(query, top_k)
    return _pt_results_to_list(res, top_k)


def display_results(results: List[Dict], title: str = 'Results') -> None:
    """Print search results as a readable table."""
    print(f'\n── {title} ({len(results)} hits) ──────────────────────────────')
    print(f"  {'#':>2}  {'Ref':<10}  {'Surah':<16}  {'Score':>9}  Text")
    print('  ' + '─' * 97)
    for i, r in enumerate(results, 1):
        ref  = f"[{r.get('surah_id','')}:{r.get('ayah_id','')}]"
        text = r.get('text', '')[:68] + ('…' if len(r.get('text','')) > 68 else '')
        print(f"  {i:>2}  {ref:<10}  {r.get('surah_name',''):<16}  {r['score']:>9.4f}  {text}")


# ── Demo ──────────────────────────────────────────────────────────────────────
demo_q = 'الله الرحمة النور'
display_results(search_or(demo_q,  top_k=10), f'TF-IDF OR  |  "{demo_q}"')
display_results(search_and('الصبر الصلاة', top_k=10), 'BM25 AND  |  "الصبر الصلاة"')


── TF-IDF OR  |  "الله الرحمة النور" (10 hits) ──────────────────────────────
   #  Ref         Surah                 Score  Text
  ─────────────────────────────────────────────────────────────────────────────────────────────────
   1  [57:9]      الحديد               7.2572  هُوَ ٱلَّذِي يُنَزِّلُ عَلَىٰ عَبۡدِهِۦٓ ءَايَٰتِۭ بَيِّنَٰتٖ لِّيُخ…
   2  [9:32]      التوبة               6.8783  يُرِيدُونَ أَن يُطۡفِـُٔواْ نُورَ ٱللَّهِ بِأَفۡوَٰهِهِمۡ وَيَأۡبَى …
   3  [61:8]      الصف                 6.6327  يُرِيدُونَ لِيُطۡفِـُٔواْ نُورَ ٱللَّهِ بِأَفۡوَٰهِهِمۡ وَٱللَّهُ مُ…
   4  [57:28]     الحديد               6.5771  يَـٰٓأَيُّهَا ٱلَّذِينَ ءَامَنُواْ ٱتَّقُواْ ٱللَّهَ وَءَامِنُواْ بِ…
   5  [35:20]     فاطر                 6.1558  وَلَا ٱلظُّلُمَٰتُ وَلَا ٱلنُّورُ
   6  [2:257]     البقرة               5.6516  ٱللَّهُ وَلِيُّ ٱلَّذِينَ ءَامَنُواْ يُخۡرِجُهُم مِّنَ ٱلظُّلُمَٰتِ …
   7  [24:35]     النور                5.5694  ۞ٱللَّهُ نُورُ ٱلسَّمَٰوَٰتِ وَٱلۡأَرۡضِۚ مَثَلُ نُورِهِ

/tmp/ipykernel_15505/707831840.py:5: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  _bm25   = pt.BatchRetrieve(index_ref, wmodel='BM25',   num_results=50)
/tmp/ipykernel_15505/707831840.py:6: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  _tfidf  = pt.BatchRetrieve(index_ref, wmodel='TF_IDF', num_results=50)
/tmp/ipykernel_15505/707831840.py:7: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  _and_br = pt.BatchRetrieve(index_ref, wmodel='BM25',   num_results=50,


---
## Section 7 — RM3 Pseudo Relevance Feedback via PyTerrier

Replaces the manual Rocchio centroid with PyTerrier's `pt.rewrite.RM3`.

**Pipeline:** `BM25 (first pass) → RM3 query expansion → BM25 (re-rank)`

| Parameter | Meaning | Default |
|-----------|---------|--------|
| `fb_docs` | pseudo-relevant docs | 5 |
| `fb_terms` | expansion terms | 5 |
| `fb_lambda` | interpolation weight | 0.75 |

In [59]:
def search_rocchio(
    query    : str,
    fb_docs  : int   = 5,
    fb_terms : int   = 5,
    fb_lambda: float = 0.75,
    top_k    : int   = 10,
) -> List[Dict]:
    """
    PRF via PyTerrier RM3: BM25 first pass → RM3 expansion → BM25 re-rank.
    Parameters mirror the original Rocchio signature for drop-in compatibility.
    """
    q_str = _preprocess_query(query)
    if not q_str.strip():
        print('Query empty after preprocessing.')
        return []

    rm3  = pt.rewrite.RM3(index_ref, fb_docs=fb_docs, fb_terms=fb_terms,
                           fb_lambda=fb_lambda)
    pipe = _bm25 >> rm3 >> _bm25

    topics = pd.DataFrame([{'qid': '1', 'query': q_str}])

    # Show expansion for transparency
    try:
        expanded_q = (_bm25 >> rm3).transform(topics)['query'].iloc[0]
        _, q_orig  = preprocess(query)
        orig_set   = set(q_orig)
        new_terms  = [t for t in expanded_q.split() if t not in orig_set]
        print(f'  Original tokens : {q_orig}')
        print(f'  Added terms     : {new_terms}')
    except Exception:
        pass

    return _pt_results_to_list(pipe.transform(topics), top_k)


# ── Demo ──────────────────────────────────────────────────────────────────────
display_results(
    search_rocchio('الصبر الصلاة', top_k=10),
    'RM3 PRF  |  "الصبر الصلاة"'
)

  Original tokens : ['صبر', 'صله']
  Added terms     : ['applypipeline:off', 'ءمن^0.038310789', 'صبر^0.461689204', 'صله^0.423378408', 'عين^0.038310789', 'ياي^0.038310789']

── RM3 PRF  |  "الصبر الصلاة" (10 hits) ──────────────────────────────
   #  Ref         Surah                 Score  Text
  ─────────────────────────────────────────────────────────────────────────────────────────────────
   1  [2:153]     البقرة              17.0909  يَـٰٓأَيُّهَا ٱلَّذِينَ ءَامَنُواْ ٱسۡتَعِينُواْ بِٱلصَّبۡرِ وَٱلصَّ…
   2  [2:45]      البقرة              13.7855  وَٱسۡتَعِينُواْ بِٱلصَّبۡرِ وَٱلصَّلَوٰةِۚ وَإِنَّهَا لَكَبِيرَةٌ إِ…
   3  [22:35]     الحج                10.7534  ٱلَّذِينَ إِذَا ذُكِرَ ٱللَّهُ وَجِلَتۡ قُلُوبُهُمۡ وَٱلصَّـٰبِرِينَ…
   4  [31:17]     لقمان               10.7534  يَٰبُنَيَّ أَقِمِ ٱلصَّلَوٰةَ وَأۡمُرۡ بِٱلۡمَعۡرُوفِ وَٱنۡهَ عَنِ ٱ…
   5  [13:22]     الرعد                9.4998  وَٱلَّذِينَ صَبَرُواْ ٱبۡتِغَآءَ وَجۡهِ رَبِّهِمۡ وَأَقَامُواْ ٱلصّ…
   6  [3:200]     آ

---
## Section 8 — BERT Semantic Query Expansion

Uses **`paraphrase-multilingual-MiniLM-L12-v2`** (supports Arabic).
Wrapped as a `pt.Transformer` subclass so it slots into `>>` pipelines.

**Pipeline:** `BERTQueryExpander → BM25`

In [60]:
from sentence_transformers import SentenceTransformer

SBERT = SentenceTransformer(
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
)

VOCAB_TERMS = [entry.getKey() for entry in lex]
print(f'Encoding {len(VOCAB_TERMS):,} vocabulary terms — may take ~30 s on CPU …')
VOCAB_VECS = SBERT.encode(
    VOCAB_TERMS, batch_size=512, show_progress_bar=True, normalize_embeddings=True
)
print('Vocabulary embeddings cached.')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 3,942 vocabulary terms — may take ~30 s on CPU …


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Vocabulary embeddings cached.


In [61]:
class BERTQueryExpander(pt.Transformer):
    """PyTerrier transformer: SBERT cosine similarity query expansion."""

    def __init__(self, k_terms: int = 5, threshold: float = 0.55):
        self.k_terms   = k_terms
        self.threshold = threshold

    def transform(self, topics_df: pd.DataFrame) -> pd.DataFrame:
        new_queries = []
        for _, row in topics_df.iterrows():
            tokens = row['query'].split()
            q_vec  = SBERT.encode(' '.join(tokens), normalize_embeddings=True)
            sims   = VOCAB_VECS @ q_vec
            q_set  = set(tokens)
            new_terms = []
            for idx in np.argsort(sims)[::-1]:
                if len(new_terms) >= self.k_terms:
                    break
                term = VOCAB_TERMS[idx]
                if term not in q_set and float(sims[idx]) >= self.threshold:
                    new_terms.append(term)
            new_queries.append(' '.join(tokens + new_terms))
        result = topics_df.copy()
        result['query'] = new_queries
        return result


_bert_expander = BERTQueryExpander(k_terms=5, threshold=0.55)
_bert_pipe     = _bert_expander >> _bm25


def search_bert(
    query    : str,
    k_terms  : int   = 5,
    threshold: float = 0.55,
    top_k    : int   = 10,
) -> List[Dict]:
    """BERT semantic expansion (SBERT cosine) → BM25 retrieval via PyTerrier pipeline."""
    q_str = _preprocess_query(query)
    if not q_str.strip():
        print('Query empty after preprocessing.')
        return []

    expander  = BERTQueryExpander(k_terms=k_terms, threshold=threshold)
    topics    = pd.DataFrame([{'qid': '1', 'query': q_str}])
    expanded  = expander.transform(topics)
    _, q_orig = preprocess(query)
    orig_set  = set(q_orig)
    new_terms = [t for t in expanded['query'].iloc[0].split() if t not in orig_set]
    print(f'  Original tokens : {q_orig}')
    print(f'  Added terms     : {new_terms}')

    return _pt_results_to_list(_bm25.transform(expanded), top_k)


# ── Demo ──────────────────────────────────────────────────────────────────────
display_results(
    search_bert('الصبر الصلاة', top_k=10),
    'BERT Expansion  |  "الصبر الصلاة"'
)

  Original tokens : ['صبر', 'صله']
  Added terms     : ['تعم', 'قفي', 'صوف', 'تعظ', 'حوذ']

── BERT Expansion  |  "الصبر الصلاة" (10 hits) ──────────────────────────────
   #  Ref         Surah                 Score  Text
  ─────────────────────────────────────────────────────────────────────────────────────────────────
   1  [2:153]     البقرة              15.8749  يَـٰٓأَيُّهَا ٱلَّذِينَ ءَامَنُواْ ٱسۡتَعِينُواْ بِٱلصَّبۡرِ وَٱلصَّ…
   2  [2:45]      البقرة              14.3340  وَٱسۡتَعِينُواْ بِٱلصَّبۡرِ وَٱلصَّلَوٰةِۚ وَإِنَّهَا لَكَبِيرَةٌ إِ…
   3  [22:46]     الحج                13.7344  أَفَلَمۡ يَسِيرُواْ فِي ٱلۡأَرۡضِ فَتَكُونَ لَهُمۡ قُلُوبٞ يَعۡقِلُو…
   4  [22:35]     الحج                11.1813  ٱلَّذِينَ إِذَا ذُكِرَ ٱللَّهُ وَجِلَتۡ قُلُوبُهُمۡ وَٱلصَّـٰبِرِينَ…
   5  [31:17]     لقمان               11.1813  يَٰبُنَيَّ أَقِمِ ٱلصَّلَوٰةَ وَأۡمُرۡ بِٱلۡمَعۡرُوفِ وَٱنۡهَ عَنِ ٱ…
   6  [58:19]     المجادلة            10.9830  ٱسۡتَحۡوَذَ عَلَيۡهِمُ ٱلشَّيۡطَٰنُ فَأَنسَىٰه

---
## Section 9 — Unified Search API & Method Comparison

In [62]:
def search(
    query  : str,
    method : str = 'or',
    top_k  : int = 10,
    **kwargs,
) -> List[Dict]:
    """
    Unified search API — entry point for Phase 3.

    Args:
        query  : Arabic query string
        method : 'or' | 'and' | 'rocchio' | 'bert'
        top_k  : number of results to return
        **kwargs: forwarded to the search function
                  rocchio — fb_docs (int), fb_terms (int), fb_lambda (float)
                  bert    — k_terms (int), threshold (float)

    Returns:
        List[Dict] with keys: doc_id, surah_id, surah_name, ayah_id, text, score
    """
    _dispatch = {
        'or'     : search_or,
        'and'    : search_and,
        'rocchio': search_rocchio,
        'bert'   : search_bert,
    }
    fn = _dispatch.get(method)
    if fn is None:
        raise ValueError(f"Unknown method '{method}'. Choose from: {list(_dispatch)}.")
    return fn(query, top_k=top_k, **kwargs)


def compare_methods(query: str, top_k: int = 5) -> None:
    """Run all four methods on the same query and display results side-by-side."""
    print(f'\n{"="*110}')
    print(f'  QUERY: "{query}"')
    print(f'{"="*110}')
    for method in ('or', 'and', 'rocchio', 'bert'):
        results = search(query, method=method, top_k=top_k)
        display_results(results, method.upper())


# ── Demo ──────────────────────────────────────────────────────────────────────
compare_methods('النور والهداية', top_k=5)


  QUERY: "النور والهداية"

── OR (5 hits) ──────────────────────────────
   #  Ref         Surah                 Score  Text
  ─────────────────────────────────────────────────────────────────────────────────────────────────
   1  [35:20]     فاطر                 6.1558  وَلَا ٱلظُّلُمَٰتُ وَلَا ٱلنُّورُ
   2  [61:8]      الصف                 5.4969  يُرِيدُونَ لِيُطۡفِـُٔواْ نُورَ ٱللَّهِ بِأَفۡوَٰهِهِمۡ وَٱللَّهُ مُ…
   3  [9:32]      التوبة               5.3238  يُرِيدُونَ أَن يُطۡفِـُٔواْ نُورَ ٱللَّهِ بِأَفۡوَٰهِهِمۡ وَيَأۡبَى …
   4  [92:12]     الليل                5.1531  إِنَّ عَلَيۡنَا لَلۡهُدَىٰ
   5  [96:11]     العلق                4.7779  أَرَءَيۡتَ إِن كَانَ عَلَى ٱلۡهُدَىٰٓ

── AND (5 hits) ──────────────────────────────
   #  Ref         Surah                 Score  Text
  ─────────────────────────────────────────────────────────────────────────────────────────────────
   1  [35:20]     فاطر                11.2301  وَلَا ٱلظُّلُمَٰتُ وَلَا ٱلنُّورُ
   2  [61:8]      ا

In [63]:
# Save BERT vocabulary embeddings for Phase 3 (avoids re-encoding)
np.save(os.path.join(OUTPUT_DIR, 'vocab_vecs.npy'), VOCAB_VECS)
with open(os.path.join(OUTPUT_DIR, 'vocab_terms.json'), 'w', encoding='utf-8') as f:
    json.dump(VOCAB_TERMS, f, ensure_ascii=False)

print(f'Saved vocab_vecs.npy    shape={VOCAB_VECS.shape}  dtype={VOCAB_VECS.dtype}')
print(f'Saved vocab_terms.json  ({len(VOCAB_TERMS):,} terms)')
print(f'\nAll Phase 2 artifacts saved to ./{OUTPUT_DIR}/')

Saved vocab_vecs.npy    shape=(3942, 384)  dtype=float32
Saved vocab_terms.json  (3,942 terms)

All Phase 2 artifacts saved to .//content/quran_ir_output/


---
## Section 10 — Professional Search Engine UI

An interactive, calligraphic search interface rendered directly in the notebook using `IPython.display`.

**Features:**
- Live search with method selector (TF-IDF OR / BM25 AND / RM3 PRF / BERT)
- Results cards with surah badge, ayah reference, score bar, and highlighted terms
- Expandable filter panel (surah type, score threshold, results count)
- Stats dashboard (query time, unique surahs, top surah)
- Full RTL Arabic layout with custom calligraphic typography

---
## Summary

### Phase 1 — Indexing
```
quran.json
  │  load_quran + validate_structure
  │  build_corpus           → 6 236 ayah documents
  │  preprocess()           → diacritics → normalize → tokenize → stopwords → ISRI stem
  └─ build_pt_index()       → PyTerrier IterDictIndexer (on-disk Terrier index)
       └─ saved artifacts: pt_index/ · c_quran.json · vocab.json · doc_metadata.json
```

### Phase 2 — Retrieval & Query Expansion
```
User query
  │  preprocess()  (identical pipeline → tokens match the index)
  │
  ├─ [or]      search_or()       pt.BatchRetrieve(TF_IDF), OR union
  ├─ [and]     search_and()      pt.BatchRetrieve(BM25, AND control)
  ├─ [rocchio] search_rocchio()  BM25 >> pt.rewrite.RM3 >> BM25
  └─ [bert]    search_bert()     BERTQueryExpander(SBERT cosine) >> BM25
          └─ unified:  search(query, method='or'|'and'|'rocchio'|'bert', top_k=10)
```

### Phase 3 — Search Engine UI
```
IPython HTML widget
  │  RTL Arabic calligraphic interface (Scheherazade New + Amiri + IBM Plex Mono)
  │  Method selector: TF-IDF OR · BM25 AND · RM3 PRF · BERT
  │  Filter panel: surah type (Meccan/Medinan) · results count
  │  Stats bar: query time · hit count · unique surahs · top result
  └─ Result cards: surah badge · ayah ref · score bar · highlighted text · pagination
```

| Function | Input | Returns |
|----------|-------|---------|
| `search_or(query)` | Arabic text | top-k results (TF-IDF OR) |
| `search_and(query)` | Arabic text | top-k results (BM25 AND) |
| `search_rocchio(query)` | Arabic text | top-k results (RM3 PRF) |
| `search_bert(query)` | Arabic text | top-k results (BERT expansion) |
| `search(query, method, top_k)` | Arabic text | unified dispatch |
| `compare_methods(query)` | Arabic text | side-by-side comparison |